In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import pandas as pd

test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

submission = pd.DataFrame({
    "ID": test["id"],
    "Prediction": ["A B C"] * len(test)
})

submission.to_csv("sample_submission.csv", index=False)

In [3]:
import pandas as pd
import numpy as np
import string
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

# Load the data
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

# ============================================
# Question 1: Frequency distribution of correct answers
# ============================================
print("=" * 50)
print("QUESTION 1: Frequency Distribution of Correct Answers")
print("=" * 50)

answer_freq = train_df['answer'].value_counts().sort_index()
print("Frequency of each answer:")
print(answer_freq)

most_frequent = answer_freq.max()
least_frequent = answer_freq.min()
sum_most_least = most_frequent + least_frequent

print(f"\nMost frequent option count: {most_frequent}")
print(f"Least frequent option count: {least_frequent}")
print(f"Sum of most and least frequent: {sum_most_least}")

# ============================================
# Question 2: Vocabulary size after cleaning prompts
# ============================================
print("\n" + "=" * 50)
print("QUESTION 2: Vocabulary Size After Cleaning")
print("=" * 50)

def clean_text(text):
    # Convert to lowercase
    text = text.lower()
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text

# Clean all prompts
train_df['cleaned_prompt'] = train_df['prompt'].apply(clean_text)

# Get all unique words
all_words = set()
for prompt in train_df['cleaned_prompt']:
    words = prompt.split()
    all_words.update(words)

vocab_size = len(all_words)
print(f"Total unique words (vocabulary size): {vocab_size}")

# ============================================
# Question 3: Words left in Row ID 1 after removing stop words
# ============================================
print("\n" + "=" * 50)
print("QUESTION 3: Words in Row ID 1 After Removing Stop Words")
print("=" * 50)

# Get cleaned prompt for Row ID 1
row1_prompt = train_df[train_df['id'] == 1]['cleaned_prompt'].values[0]

# Split into words
row1_words = row1_prompt.split()

# Filter out stop words
row1_filtered = [word for word in row1_words if word not in ENGLISH_STOP_WORDS]

words_left = len(row1_filtered)
print(f"Row ID 1 cleaned prompt (first 100 chars): {row1_prompt[:100]}...")
print(f"Words left after removing stop words: {words_left}")

# ============================================
# Question 4: TF-IDF Vectorizer vocabulary size
# ============================================
print("\n" + "=" * 50)
print("QUESTION 4: TF-IDF Vectorizer Vocabulary Size")
print("=" * 50)

# Combine prompt and all options into single documents for each row
combined_texts = []
for idx, row in train_df.iterrows():
    # Combine prompt with all options
    combined = row['prompt'] + ' ' + row['A'] + ' ' + row['B'] + ' ' + row['C'] + ' ' + row['D'] + ' ' + row['E']
    combined_texts.append(combined)

# Fit TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_vectorizer.fit(combined_texts)

feature_columns = len(tfidf_vectorizer.get_feature_names_out())
print(f"Number of feature columns (vocabulary size): {feature_columns}")

# ============================================
# Question 5: Cosine similarity between prompt and option A for Row ID 1
# ============================================
print("\n" + "=" * 50)
print("QUESTION 5: Cosine Similarity for Row ID 1 (Prompt vs Option A)")
print("=" * 50)

# Get Row ID 1 data
row1 = train_df[train_df['id'] == 1].iloc[0]

# Transform prompt and option A separately
prompt_vector = tfidf_vectorizer.transform([row1['prompt']])
option_a_vector = tfidf_vectorizer.transform([row1['A']])

# Calculate cosine similarity
similarity_score = cosine_similarity(prompt_vector, option_a_vector)[0][0]

print(f"Prompt: {row1['prompt'][:100]}...")
print(f"Option A: {row1['A'][:100]}...")
print(f"Cosine similarity score: {similarity_score:.4f}")

# ============================================
# Question 6: Percentage where highest similarity matches correct answer
# ============================================
print("\n" + "=" * 50)
print("QUESTION 6: Percentage of Highest Similarity Matching Correct Answer")
print("=" * 50)

correct_matches = 0
total_rows = len(train_df)

for idx, row in train_df.iterrows():
    # Vectorize prompt
    prompt_vec = tfidf_vectorizer.transform([row['prompt']])
    
    # Vectorize each option and calculate similarity
    similarities = {}
    for option in ['A', 'B', 'C', 'D', 'E']:
        option_vec = tfidf_vectorizer.transform([row[option]])
        sim = cosine_similarity(prompt_vec, option_vec)[0][0]
        similarities[option] = sim
    
    # Find option with highest similarity
    highest_sim_option = max(similarities, key=similarities.get)
    
    # Check if matches correct answer
    if highest_sim_option == row['answer']:
        correct_matches += 1

percentage = (correct_matches / total_rows) * 100
print(f"Rows where highest similarity matches correct answer: {correct_matches}/{total_rows}")
print(f"Percentage: {percentage:.2f}%")

# ============================================
# Question 7: MAP@3 score for prediction C A B when answer is C
# ============================================
print("\n" + "=" * 50)
print("QUESTION 7: MAP@3 for prediction C A B (answer is C)")
print("=" * 50)

def calculate_map_at_3(ground_truth, predictions):
    """
    Calculate MAP@3 for a single question
    predictions: list of 3 predicted answers in order
    """
    for i, pred in enumerate(predictions):
        if pred == ground_truth:
            return 1.0 / (i + 1)  # 1/k where k is the position (1-indexed)
    return 0.0  # Not in top 3

# Example: answer is C, prediction is C A B
map_score_q7 = calculate_map_at_3('C', ['C', 'A', 'B'])
print(f"Ground truth: C, Prediction: C A B")
print(f"MAP@3 score: {map_score_q7}")

# ============================================
# Question 8: MAP@3 score for prediction D B E when answer is B
# ============================================
print("\n" + "=" * 50)
print("QUESTION 8: MAP@3 for prediction D B E (answer is B)")
print("=" * 50)

map_score_q8 = calculate_map_at_3('B', ['D', 'B', 'E'])
print(f"Ground truth: B, Prediction: D B E")
print(f"MAP@3 score: {map_score_q8}")

# ============================================
# Question 9: Majority Class Baseline MAP@3
# ============================================
print("\n" + "=" * 50)
print("QUESTION 9: Majority Class Baseline MAP@3")
print("=" * 50)

# Get frequency of answers
answer_counts = train_df['answer'].value_counts()
print("Answer frequencies:")
print(answer_counts)

# Get top 3 most frequent answers
top3_answers = answer_counts.head(3).index.tolist()
print(f"Top 3 most frequent answers: {top3_answers}")

# Calculate MAP@3 for majority baseline
majority_scores = []
for idx, row in train_df.iterrows():
    ground_truth = row['answer']
    predictions = top3_answers  # Always predict the same top 3
    score = calculate_map_at_3(ground_truth, predictions)
    majority_scores.append(score)

overall_majority_map = np.mean(majority_scores)
print(f"Overall MAP@3 for Majority Class Baseline: {overall_majority_map:.4f}")

# ============================================
# Question 10: TF-IDF Pipeline MAP@3
# ============================================
print("\n" + "=" * 50)
print("QUESTION 10: TF-IDF Pipeline MAP@3")
print("=" * 50)

tfidf_scores = []

for idx, row in train_df.iterrows():
    # Vectorize prompt
    prompt_vec = tfidf_vectorizer.transform([row['prompt']])
    
    # Calculate similarity for each option
    similarities = {}
    for option in ['A', 'B', 'C', 'D', 'E']:
        option_vec = tfidf_vectorizer.transform([row[option]])
        sim = cosine_similarity(prompt_vec, option_vec)[0][0]
        similarities[option] = sim
    
    # Sort options by similarity (highest to lowest)
    sorted_options = sorted(similarities.items(), key=lambda x: x[1], reverse=True)
    
    # Get top 3 predictions
    top3_predictions = [option for option, sim in sorted_options[:3]]
    
    # Calculate MAP@3 for this row
    ground_truth = row['answer']
    score = calculate_map_at_3(ground_truth, top3_predictions)
    tfidf_scores.append(score)

overall_tfidf_map = np.mean(tfidf_scores)
print(f"Overall MAP@3 for TF-IDF Pipeline: {overall_tfidf_map:.4f}")

# ============================================
# Create sample submission file
# ============================================
print("\n" + "=" * 50)
print("Creating Sample Submission File")
print("=" * 50)

# Create predictions for test set using TF-IDF approach
submission_predictions = []

for idx, row in test_df.iterrows():
    prompt_vec = tfidf_vectorizer.transform([row['prompt']])
    
    similarities = {}
    for option in ['A', 'B', 'C', 'D', 'E']:
        option_vec = tfidf_vectorizer.transform([row[option]])
        sim = cosine_similarity(prompt_vec, option_vec)[0][0]
        similarities[option] = sim
    
    sorted_options = sorted(similarities.items(), key=lambda x: x[1], reverse=True)
    top3_predictions = [option for option, sim in sorted_options[:3]]
    
    submission_predictions.append({
        'ID': row['id'],
        'Prediction': ' '.join(top3_predictions)
    })

# Create submission DataFrame
submission_df = pd.DataFrame(submission_predictions)

# Save to CSV
submission_df.to_csv('sample_submission.csv', index=False)

print(f"Sample submission file created with {len(submission_df)} predictions")
print("\nFirst few predictions:")
print(submission_df.head())

QUESTION 1: Frequency Distribution of Correct Answers
Frequency of each answer:
answer
A    369
B    490
C    459
D    358
E    324
Name: count, dtype: int64

Most frequent option count: 490
Least frequent option count: 324
Sum of most and least frequent: 814

QUESTION 2: Vocabulary Size After Cleaning
Total unique words (vocabulary size): 859

QUESTION 3: Words in Row ID 1 After Removing Stop Words
Row ID 1 cleaned prompt (first 100 chars): pick the best possible answer what is martin heideggers view on the relationship between time and hu...
Words left after removing stop words: 13

QUESTION 4: TF-IDF Vectorizer Vocabulary Size
Number of feature columns (vocabulary size): 2762

QUESTION 5: Cosine Similarity for Row ID 1 (Prompt vs Option A)
Prompt: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and ...
Option A: Martin Heidegger believes that humans exist within a time continuum that is infinite and does not ha...
Cosine similarity sco